# [가치 입증] 생성 결과 평가 (G-Eval 기반)

이 노트북은 이전 단계(`07_inference_generation.ipynb`)에서 생성한 3가지 전략의 교정 결과물(1,156건) 중 **100건을 층화 샘플링(Stratified Sampling)하여 LLM-as-a-Judge(G-Eval)로 블라인드 채점**한 결과를 분석합니다.

이를 통해 **SmartRouter가 대형 모델 대비 품질을 얼마나 방어하면서, 비용을 절감하는지 정량적으로 증명**합니다.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'Malgun Gothic' # Windows 한글 폰트
plt.rcParams['axes.unicode_minus'] = False

## 1. G-Eval 채점 결과 로드

100건의 샘플에 대해 대형 모델(심사위원)이 5점 만점으로 채점한 결과입니다.

> **평가 기준 (Rubric)**
> 1. **Intent Accuracy**: 강도(Intensity)를 잘 반영했는가?
> 2. **Context Appropriateness**: 도메인(Field)에 맞는 톤앤매너인가?
> 3. **Quality**: 문법 오류 없이 유창한가?

In [2]:
df_eval = pd.read_csv('../data/8. evaluation_results.csv')
print(f"평가 샘플 수: {len(df_eval)}건")
df_eval[['source_text', 'score_A', 'score_B', 'score_C']].head()

## 2. 품질-비용 시각화 분석 (핵심 지표)

In [3]:
mean_scores = [df_eval['score_A'].mean(), df_eval['score_B'].mean(), df_eval['score_C'].mean()]
total_costs = [df_eval['cost_A'].sum(), df_eval['cost_B'].sum(), df_eval['cost_C'].sum()]
labels = ['전략 A\n(Nano 단독)', '전략 B\n(Mini 단독)', '전략 C\n(SmartRouter)']
colors = ['#FF9999', '#66B2FF', '#99FF99']

fig, ax1 = plt.subplots(figsize=(10, 6))

# 막대 그래프 (점수)
bars = ax1.bar(labels, mean_scores, color=colors, alpha=0.7, edgecolor='black')
ax1.set_ylabel('평균 G-Eval 점수 (5점 만점)', fontsize=12, fontweight='bold')
ax1.set_ylim(4.0, 4.5)

for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + 0.01, round(yval, 2), ha='center', va='bottom', fontsize=12, fontweight='bold')

# 꺾은선 그래프 (비용)
ax2 = ax1.twinx()
line = ax2.plot(labels, total_costs, color='#FF3333', marker='o', linestyle='-', linewidth=2, markersize=8)
ax2.set_ylabel('총 추론 비용 ($)', fontsize=12, fontweight='bold', color='#FF3333')
ax2.set_ylim(0, max(total_costs)*1.2)

for i, cost in enumerate(total_costs):
    ax2.text(i, cost + 0.0002, f"${cost:.4f}", ha='center', va='bottom', color='#FF3333', fontweight='bold')

plt.title('교정 전략별 품질(점수) vs 비용 비교', fontsize=16, fontweight='bold', pad=20)
plt.show()

## 3. 최종 결론 산출

In [4]:
retention = (mean_scores[2] / mean_scores[1]) * 100
cost_saving = (1 - total_costs[2] / total_costs[1]) * 100

print("=" * 50)
print("✨ [SmartRouter 최종 성과 지표] ✨")
print("=" * 50)
print(f"📌 대형 모델(Mini) 단독 대비 품질 유지율 : {retention:.1f}% 방어 성공")
print(f"📌 대형 모델(Mini) 단독 대비 비용 절감율 : {cost_saving:.1f}% 절감 성공")
print("=" * 50)